In [1]:
!pip install --quiet -U torch torchvision cython
!pip install --quiet -U 'git+https://github.com/facebookresearch/fvcore.git' 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
!pip install --quiet -U 'detectron2@git+https://github.com/facebookresearch/detectron2.git@main'
!pip install --quiet fiftyone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

In [ ]:
import fiftyone.zoo as foz

training_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="train",
    label_types=["detections", "segmentations"],
    max_samples=5000,
)

validation_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",
    label_types=["detections", "segmentations"],
    max_samples=625,
)

'''
testing_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="test",
    label_types=["detections", "segmentations"],
    max_samples=625,
)
'''

In [ ]:
from detectron2.data.datasets import register_coco_instances
register_coco_instances("coco_train", {}, "/root/fiftyone/coco-2017/train/labels.json", "/root/fiftyone/coco-2017/train/data")
register_coco_instances("coco_val", {}, "/root/fiftyone/coco-2017/validation/labels.json", "/root/fiftyone/coco-2017/validation/data")
#register_coco_instances("coco_test", {}, "/root/fiftyone/coco-2017/test/labels.json", "/root/fiftyone/coco-2017/test/data")

In [4]:
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.model_zoo import get_checkpoint_url
from detectron2 import model_zoo

BATCH_SIZE = 16

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/fast_rcnn_R_50_FPN_1x.yaml"))
cfg.MODEL.WEIGHTS = ""
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 80
cfg.MODEL.DEVICE = device

cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
cfg.SOLVER.MAX_ITER = int(5000/BATCH_SIZE * 100) # Convert epochs to iteractions, since epoch is not a config option
cfg.TEST.EVAL_PERIOD = int(5000/BATCH_SIZE)  # Evaluate every epoch (5000/64)

cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.CLIP_GRADIENTS.ENABLED = True
cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "norm"
cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE = 1.0

cfg.DATASETS.TRAIN = ("coco_train",)
cfg.DATASETS.TEST = ("coco_val",) # Validation

In [5]:
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator

class Trainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = cfg.OUTPUT_DIR
        return COCOEvaluator(dataset_name, cfg, False, output_folder)

In [ ]:
import os

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

model = Trainer(cfg)
model.resume_or_load(resume=True)
model.train()

[06/02 02:25:54 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

coco_2017_val_box_proposals_ee0dad.pkl: 136MB [00:07, 19.3MB/s]                           


Streaming output truncated to the last 5000 lines.
[06/02 05:03:06 d2.utils.events]:  eta: 2:59:43  iter: 14199  total_loss: 1.216  loss_cls: 0.7203  loss_box_reg: 0.477    time: 0.6093  last_time: 0.6995  data_time: 0.1888  last_data_time: 0.2891   lr: 0.00025  max_mem: 13949M
[06/02 05:03:18 d2.utils.events]:  eta: 2:59:31  iter: 14219  total_loss: 1.131  loss_cls: 0.6949  loss_box_reg: 0.4489    time: 0.6093  last_time: 0.5179  data_time: 0.1796  last_data_time: 0.0760   lr: 0.00025  max_mem: 13949M
[06/02 05:03:30 d2.utils.events]:  eta: 2:59:18  iter: 14239  total_loss: 1.09  loss_cls: 0.6554  loss_box_reg: 0.4342    time: 0.6093  last_time: 0.6238  data_time: 0.1876  last_data_time: 0.2257   lr: 0.00025  max_mem: 13949M
[06/02 05:03:42 d2.utils.events]:  eta: 2:58:58  iter: 14259  total_loss: 1.145  loss_cls: 0.6904  loss_box_reg: 0.4634    time: 0.6093  last_time: 0.6533  data_time: 0.1786  last_data_time: 0.2400   lr: 0.00025  max_mem: 13949M
[06/02 05:03:54 d2.utils.events]:  

In [7]:
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/fast_rcnn_R_50_FPN_1x.yaml"))
#cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 80
cfg.MODEL.DEVICE = device

model = build_model(cfg)
DetectionCheckpointer(model).load("output/model_final.pth")

{'trainer': {'iteration': 31249,
  'hooks': {'LRScheduler': {'base_lrs': [0.00025], 'last_epoch': 31250}},
  '_trainer': {'iteration': 31249,
   'optimizer': {'state': {0: {'momentum_buffer': tensor([[[[ 1.7334e-05]],
      
               [[ 2.2105e-03]],
      
               [[ 5.9981e-03]],
      
               ...,
      
               [[ 3.0442e-03]],
      
               [[ 6.8225e-03]],
      
               [[ 3.0886e-03]]],
      
      
              [[[-2.6998e-05]],
      
               [[-1.3762e-03]],
      
               [[-3.9069e-03]],
      
               ...,
      
               [[-2.2980e-03]],
      
               [[-5.0374e-03]],
      
               [[-3.3646e-03]]],
      
      
              [[[ 6.7490e-05]],
      
               [[-1.7073e-03]],
      
               [[ 1.1429e-04]],
      
               ...,
      
               [[ 5.9475e-04]],
      
               [[ 3.3277e-03]],
      
               [[ 2.2949e-04]]],
      
      
       

In [9]:
from detectron2.data.datasets import register_coco_instances
import fiftyone as fo
import fiftyone.zoo as foz
#Use latter half of validation dataset for training since test does not contain labels

testing_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",
    label_types=["detections", "segmentations"],
    max_samples=1250,
)

testing_dataset = testing_dataset.skip(625)

testing_dataset.export(
    export_dir="/root/fiftyone/coco-2017/test",
    dataset_type=fo.types.COCODetectionDataset,
)

register_coco_instances("coco_test", {}, "/root/fiftyone/coco-2017/test/labels.json", "/root/fiftyone/coco-2017/test/data")

INFO:fiftyone.zoo.datasets:Downloading split 'validation' to '/root/fiftyone/coco-2017/validation' if necessary


INFO:fiftyone.utils.coco:Downloading annotations to '/root/fiftyone/coco-2017/tmp-download/annotations_trainval2017.zip'


 100% |██████|    1.9Gb/1.9Gb [16.5s elapsed, 0s remaining, 141.8Mb/s]      


INFO:eta.core.utils: 100% |██████|    1.9Gb/1.9Gb [16.5s elapsed, 0s remaining, 141.8Mb/s]      


Extracting annotations to '/root/fiftyone/coco-2017/raw/instances_val2017.json'


...

INFO:fiftyone.zoo.datasets:Dataset 'coco-2017-validation-1250' created


Found multiple fields ['detections', 'segmentations'] with compatible type (<class 'fiftyone.core.labels.Detections'>, <class 'fiftyone.core.labels.Polylines'>, <class 'fiftyone.core.labels.Keypoints'>); exporting 'detections'


INFO:fiftyone.core.collections:Found multiple fields ['detections', 'segmentations'] with compatible type (<class 'fiftyone.core.labels.Detections'>, <class 'fiftyone.core.labels.Polylines'>, <class 'fiftyone.core.labels.Keypoints'>); exporting 'detections'


 100% |█████████████████| 625/625 [3.9s elapsed, 0s remaining, 150.6 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 625/625 [3.9s elapsed, 0s remaining, 150.6 samples/s]      


In [12]:
cfg.DATASETS.TEST = ("coco_test",)

evaluator = COCOEvaluator("coco_test", cfg, False, cfg.OUTPUT_DIR)
results = Trainer.test(cfg, model, evaluators=[evaluator])
results

Loading and preparing results...
DONE (t=0.43s)
creating index...
index created!
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.016
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.042
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.009
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.010
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.013
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.023
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.044
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.089
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.097
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.040
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.078
 Average Recall     (AR) @[ IoU=0.50:0.

OrderedDict([('bbox',
              {'AP': 1.5777418897877622,
               'AP50': 4.214288953617892,
               'AP75': 0.8867579438334315,
               'APs': 1.0154996256735391,
               'APm': 1.32620565129443,
               'APl': 2.329579322538505,
               'AP-person': 7.148451908702151,
               'AP-bicycle': 0.2964255856722772,
               'AP-car': 4.165987028325267,
               'AP-motorcycle': 1.6139123243181883,
               'AP-airplane': 2.047534663690178,
               'AP-bus': 2.821772206487183,
               'AP-train': 2.5629010558276217,
               'AP-truck': 1.5688599118911633,
               'AP-boat': 0.17181663221267182,
               'AP-traffic light': 1.9312843052975683,
               'AP-fire hydrant': 1.6853685368536855,
               'AP-stop sign': 13.131552564123405,
               'AP-parking meter': 0.1370906321401371,
               'AP-bench': 0.03911356257460433,
               'AP-bird': 0.102688861390